# Herramientas y visualización de datos - Actividad práctica final
## Visualización de grandes volúmenes de datos con Apache Spark

**Dataset:** NYC Yellow Taxi Trip Records - Enero de 2024  
**Archivo:** `yellow_tripdata_2024-01.parquet`  
**Integrantes:** Escribir aquí los nombres completos  
**Fecha:** Escribir fecha de entrega

## 1. Introducción

El presente notebook desarrolla un análisis exploratorio y visual del conjunto de datos **NYC Yellow Taxi Trip Records**, correspondiente a los viajes realizados por taxis amarillos en la ciudad de Nueva York durante enero de 2024.

Este dataset contiene información real sobre movilidad urbana, incluyendo fechas y horas de recogida y llegada, duración del viaje, distancia recorrida, número de pasajeros, tarifas, propinas, tipo de pago y valores totales pagados.

La elección de este dataset se justifica porque permite analizar patrones de demanda, comportamiento temporal, relación entre distancia y costo, distribución de pagos y hábitos de movilidad. Además, al tratarse de un conjunto de datos con gran volumen de registros, resulta pertinente emplear **Apache Spark** para realizar la carga, limpieza, transformación y agregación de datos antes de generar visualizaciones con librerías de Python.

**Nota importante:** la actividad solicita un dataset mayor a 1 GB. En este notebook se incluye una celda que valida el tamaño del archivo. Si el archivo individual no alcanza 1 GB, se recomienda descargar varios meses del mismo dataset y cargarlos juntos con Spark para cumplir estrictamente el requisito de tamaño.

### Preguntas de análisis

1. ¿En qué horas del día se concentra la mayor cantidad de viajes?
2. ¿Qué tipo de pago es más utilizado por los usuarios?
3. ¿Cómo se distribuyen las distancias recorridas por los taxis?
4. ¿Existe relación entre la distancia recorrida y el valor total pagado?
5. ¿Cómo se comporta la demanda de viajes según el día de la semana y la hora del día?

## 2. Instalación y configuración inicial

Esta sección instala las librerías necesarias y prepara la configuración para ejecutar Spark en Jupyter Notebook. En Windows se agrega una configuración básica para evitar errores relacionados con Hadoop.

In [ ]:
# Instalar librerías necesarias
# Ejecutar esta celda si el entorno no tiene las librerías instaladas

%pip install pyspark pandas matplotlib seaborn plotly pyarrow -q

In [ ]:
import os
import sys
import tempfile
from pathlib import Path

# Configuración recomendada para Windows
if sys.platform.startswith('win'):
    hadoop_home = os.path.join(tempfile.gettempdir(), 'hadoop')
    os.makedirs(os.path.join(hadoop_home, 'bin'), exist_ok=True)
    os.environ['HADOOP_HOME'] = hadoop_home

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType

sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams['figure.figsize'] = (11, 6)
plt.rcParams['axes.titlesize'] = 15
plt.rcParams['axes.labelsize'] = 12

## 3. Creación de la sesión de Spark

Se crea una sesión de Spark con una configuración de memoria apropiada para procesar archivos grandes.

In [ ]:
spark = (SparkSession.builder
         .appName('NYC Yellow Taxi Spark Visualization')
         .config('spark.driver.memory', '6g')
         .config('spark.sql.shuffle.partitions', '8')
         .getOrCreate())

spark

## 4. Carga del dataset con Spark

Se carga el archivo en formato Parquet directamente con Spark. Este formato es adecuado para análisis de grandes volúmenes porque conserva tipos de datos y permite lecturas eficientes.

In [ ]:
# Ruta del archivo
# Si se ejecuta en otro equipo, ubicar el archivo en la misma carpeta del notebook o ajustar la ruta.

DATA_PATH = 'yellow_tripdata_2024-01.parquet'

# Ruta alternativa para Google Colab o entorno local
if not Path(DATA_PATH).exists():
    DATA_PATH = '/mnt/data/yellow_tripdata_2024-01.parquet'

file_size_mb = Path(DATA_PATH).stat().st_size / (1024 ** 2)
file_size_gb = Path(DATA_PATH).stat().st_size / (1024 ** 3)

print(f'Tamaño del archivo: {file_size_mb:.2f} MB ({file_size_gb:.2f} GB)')

if file_size_gb < 1:
    print('Advertencia: este archivo individual no supera 1 GB. Para cumplir estrictamente la actividad, descargar y cargar varios meses del mismo dataset.')

In [ ]:
df = spark.read.parquet(DATA_PATH)

print('Schema del dataset:')
df.printSchema()

total_registros = df.count()
print(f'Total de registros cargados: {total_registros:,}')

df.show(5, truncate=False)

## 5. Exploración inicial de los datos

Antes de visualizar se revisan valores nulos, duplicados, tipos de datos, rangos y valores únicos de columnas clave.

In [ ]:
# Valores nulos por columna
null_counts = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
])

null_counts.show(truncate=False)

In [ ]:
# Revisión de duplicados
total_duplicados = total_registros - df.dropDuplicates().count()
print(f'Total de registros duplicados: {total_duplicados:,}')

In [ ]:
# Estadísticas de columnas numéricas clave
columnas_numericas = [
    'passenger_count', 'trip_distance', 'fare_amount',
    'tip_amount', 'total_amount'
]

columnas_existentes = [c for c in columnas_numericas if c in df.columns]
df.select(columnas_existentes).describe().show()

In [ ]:
# Valores únicos de columnas categóricas clave
for col in ['payment_type', 'VendorID', 'RatecodeID']:
    if col in df.columns:
        print(f'Valores únicos de {col}:')
        df.groupBy(col).count().orderBy(F.desc('count')).show(20)

## 6. Limpieza y preparación de datos

A partir de la exploración inicial, se aplican reglas de limpieza para conservar registros lógicos y útiles para el análisis:

- Eliminación de duplicados.
- Eliminación de registros sin fechas de recogida o llegada.
- Filtrado de distancias negativas o iguales a cero.
- Filtrado de valores totales negativos o iguales a cero.
- Conservación de viajes con duración positiva.
- Creación de variables temporales: hora, día del mes, día de la semana y duración del viaje en minutos.

Estas decisiones permiten reducir ruido y evitar que registros atípicos o inválidos distorsionen las visualizaciones.

In [ ]:
df_clean = df.dropDuplicates()

# Identificar columnas de fecha del dataset de taxis amarillos
pickup_col = 'tpep_pickup_datetime'
dropoff_col = 'tpep_dropoff_datetime'

df_clean = (df_clean
    .filter(F.col(pickup_col).isNotNull())
    .filter(F.col(dropoff_col).isNotNull())
    .filter(F.col('trip_distance').isNotNull())
    .filter(F.col('total_amount').isNotNull())
    .filter(F.col('trip_distance') > 0)
    .filter(F.col('total_amount') > 0)
    .withColumn('trip_duration_min', (F.unix_timestamp(F.col(dropoff_col)) - F.unix_timestamp(F.col(pickup_col))) / 60)
    .filter(F.col('trip_duration_min') > 0)
    .filter(F.col('trip_duration_min') <= 240)
    .filter(F.col('trip_distance') <= 100)
    .filter(F.col('total_amount') <= 500)
    .withColumn('pickup_hour', F.hour(F.col(pickup_col)))
    .withColumn('pickup_day', F.dayofmonth(F.col(pickup_col)))
    .withColumn('pickup_dayofweek', F.date_format(F.col(pickup_col), 'E'))
    .withColumn('pickup_date', F.to_date(F.col(pickup_col)))
)

print(f'Registros originales: {total_registros:,}')
print(f'Registros después de limpieza: {df_clean.count():,}')

df_clean.select(pickup_col, dropoff_col, 'trip_distance', 'trip_duration_min', 'total_amount', 'payment_type').show(5)

## 7. Preparación para visualización

Las agregaciones pesadas se realizan con Spark. Solo después de reducir los datos se convierte el resultado a Pandas para graficar.

## Visualización 1: Cantidad de viajes por hora del día

Esta visualización analiza en qué horas se concentra la mayor demanda de taxis amarillos.

In [ ]:
viajes_por_hora = (df_clean
    .groupBy('pickup_hour')
    .agg(F.count('*').alias('total_viajes'))
    .orderBy('pickup_hour')
)

pd_hora = viajes_por_hora.toPandas()

plt.figure(figsize=(12, 6))
sns.lineplot(data=pd_hora, x='pickup_hour', y='total_viajes', marker='o')
plt.title('Cantidad de viajes por hora del día')
plt.xlabel('Hora del día')
plt.ylabel('Total de viajes')
plt.xticks(range(0, 24))
plt.tight_layout()
plt.show()

**Interpretación:** esta gráfica permite identificar las horas de mayor y menor demanda. Normalmente, los picos se relacionan con horarios laborales, desplazamientos urbanos, actividades nocturnas y dinámicas propias de la ciudad.

## Visualización 2: Tipos de pago más utilizados

Esta visualización compara la cantidad de viajes según el método de pago registrado.

In [ ]:
payment_labels = {
    1: 'Tarjeta de crédito',
    2: 'Efectivo',
    3: 'Sin cargo',
    4: 'Disputa',
    5: 'Desconocido',
    6: 'Viaje anulado'
}

pagos = (df_clean
    .groupBy('payment_type')
    .agg(F.count('*').alias('total_viajes'))
    .orderBy(F.desc('total_viajes'))
)

pd_pagos = pagos.toPandas()
pd_pagos['payment_label'] = pd_pagos['payment_type'].map(payment_labels).fillna('Otro')

plt.figure(figsize=(10, 6))
sns.barplot(data=pd_pagos, x='total_viajes', y='payment_label')
plt.title('Tipos de pago más utilizados en los viajes')
plt.xlabel('Total de viajes')
plt.ylabel('Tipo de pago')
plt.tight_layout()
plt.show()

**Interpretación:** el gráfico permite reconocer los medios de pago dominantes. Esta información es útil para comprender hábitos de consumo y comportamiento de los usuarios del servicio de taxi.

## Visualización 3: Distribución de la distancia recorrida

En lugar de convertir todo el dataset a Pandas, se crean intervalos de distancia con Spark y luego se grafica el conteo por rango.

In [ ]:
# Crear rangos de distancia de 0 a 30 millas
df_dist = df_clean.filter(F.col('trip_distance') <= 30)

dist_bins = (df_dist
    .withColumn('distance_bin', F.floor(F.col('trip_distance')))
    .groupBy('distance_bin')
    .agg(F.count('*').alias('total_viajes'))
    .orderBy('distance_bin')
)

pd_dist = dist_bins.toPandas()

plt.figure(figsize=(12, 6))
sns.barplot(data=pd_dist, x='distance_bin', y='total_viajes')
plt.title('Distribución de viajes por distancia recorrida')
plt.xlabel('Distancia aproximada del viaje en millas')
plt.ylabel('Total de viajes')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Interpretación:** esta visualización muestra si la mayoría de viajes son cortos, medios o largos. En servicios urbanos de taxi suele predominar una alta cantidad de trayectos de corta distancia.

## Visualización 4: Relación entre distancia y valor total pagado

Se agrupa la distancia por intervalos y se calcula el valor promedio pagado para analizar la relación entre distancia recorrida y costo del servicio.

In [ ]:
distancia_costo = (df_clean
    .filter((F.col('trip_distance') > 0) & (F.col('trip_distance') <= 40))
    .withColumn('distance_bin', F.floor(F.col('trip_distance')))
    .groupBy('distance_bin')
    .agg(
        F.avg('total_amount').alias('promedio_total_pagado'),
        F.count('*').alias('total_viajes')
    )
    .filter(F.col('total_viajes') >= 50)
    .orderBy('distance_bin')
)

pd_dist_costo = distancia_costo.toPandas()

plt.figure(figsize=(12, 6))
sns.scatterplot(
    data=pd_dist_costo,
    x='distance_bin',
    y='promedio_total_pagado',
    size='total_viajes',
    sizes=(40, 300),
    legend=True
)
plt.title('Relación entre distancia recorrida y valor promedio pagado')
plt.xlabel('Distancia aproximada en millas')
plt.ylabel('Valor promedio pagado en dólares')
plt.tight_layout()
plt.show()

**Interpretación:** la relación esperada es positiva: a mayor distancia, mayor valor promedio pagado. Sin embargo, también pueden existir variaciones por peajes, recargos, propinas, tráfico o tarifas especiales.

## Visualización 5: Mapa de calor de viajes por día de la semana y hora

Esta visualización permite observar la intensidad de la demanda cruzando dos variables temporales: día de la semana y hora del día.

In [ ]:
orden_dias = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

heatmap_data = (df_clean
    .groupBy('pickup_dayofweek', 'pickup_hour')
    .agg(F.count('*').alias('total_viajes'))
)

pd_heat = heatmap_data.toPandas()
pd_heat['pickup_dayofweek'] = pd.Categorical(pd_heat['pickup_dayofweek'], categories=orden_dias, ordered=True)

matriz_heat = pd_heat.pivot(index='pickup_dayofweek', columns='pickup_hour', values='total_viajes').fillna(0)

plt.figure(figsize=(14, 6))
sns.heatmap(matriz_heat, cmap='viridis')
plt.title('Mapa de calor de viajes por día de la semana y hora')
plt.xlabel('Hora del día')
plt.ylabel('Día de la semana')
plt.tight_layout()
plt.show()

**Interpretación:** el mapa de calor permite identificar patrones de demanda según día y hora. Las zonas más intensas indican momentos de mayor actividad del servicio, útiles para analizar movilidad, operación y disponibilidad de taxis.

## 8. Conclusiones

A partir del análisis realizado con Apache Spark se pueden plantear las siguientes conclusiones:

1. El dataset de taxis amarillos permite estudiar patrones reales de movilidad urbana en Nueva York.
2. Spark facilita la carga, limpieza y agregación de grandes volúmenes de datos sin convertir todo el archivo directamente a Pandas.
3. Las visualizaciones muestran diferencias importantes según la hora del día, el tipo de pago, la distancia recorrida y el comportamiento temporal de la demanda.
4. La limpieza de datos fue necesaria para eliminar registros con valores nulos, distancias inválidas, costos negativos o duraciones incoherentes.
5. El uso correcto del flujo Spark a Pandas permite generar gráficos profesionales sin saturar la memoria del equipo.

### Limitaciones encontradas

- El archivo usado en este notebook puede no superar 1 GB de forma individual. Para cumplir completamente la instrucción de la actividad, se recomienda trabajar con varios meses del mismo dataset o con un archivo consolidado mayor a 1 GB.
- Algunos códigos de zonas o pagos requieren documentación adicional para interpretarse con mayor precisión.
- Los datos representan únicamente taxis amarillos, por lo que no explican toda la movilidad de Nueva York.

## 9. Cierre de la sesión Spark

Al finalizar el trabajo se cierra la sesión de Spark para liberar recursos.

In [ ]:
spark.stop()
print('Sesión de Spark cerrada correctamente.')